# 00 - Pipeline complet : de la liste de sociétés au graphique PEG × ROIC

Ce notebook regroupe en une seule exécution les étapes 01 à 05 de l'analyse :

| Étape | Ce que ça fait | Sortie |
|---|---|---|
| 1 | Exploration du fichier brut | — |
| 2 | Nettoyage des tickers, exclusion des sociétés non cotées, fusion par ticker | `stocks_cleaned.csv`, `tickers.csv` |
| 3 | Résolution des tickers sur yfinance (suffixes de place de marché) | `stocks_transformed_verified.csv` |
| 4 | Calcul des KPIs (PEG Ratio (1Y) et ROIC (%)), filtrage des valeurs aberrantes et des données manquantes, puis classification en quadrants (ES / EV / PS / PV) | `stocks_kpis.csv` |
| 5 | Scatter plot interactif PEG × ROIC, coloré par quadrant | — |

> **Exécution complète en une fois :** `Kernel > Restart & Run All`.

⚠️ Les étapes 3 et 4 font des appels réseau vers yfinance (~100 requêtes chacune) : comptez **5 à 10 minutes** pour l'ensemble. Les notebooks `01` à `05` restent disponibles pour exécuter le pipeline pas à pas.

## Imports et configuration

In [ ]:
import pandas as pd
import yfinance as yf
import plotly.express as px

from pathlib import Path

# Chemins relatifs à ce notebook (notebooks/Stocks Analysis/)
RAW_CSV = Path("../../data/raw/stocks/stocks.csv")
OUT_DIR = Path("../../data/processed/stocks")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV

## Étape 1 — Exploration

Le fichier brut contient **139 sociétés** issues de la recherche sectorielle. Première inspection : taille, types, valeurs nulles, et un premier regard sur les tickers à nettoyer.

In [ ]:
# Charger le fichier CSV dans un DataFrame

df = pd.read_csv(RAW_CSV)

In [ ]:
df.head(10)

In [ ]:
df["ticker"].unique()

I notice that some tickers will need cleaning:
- remove the stock exchange, when mentioned.
- only keep 1 ticker if there are multiple values.
- remove ticker that says "Private" as no data is available.
- Only keep the Layer number in the Layer column.

In [ ]:
# Number of tickers not publicly traded
(df["ticker"] == "Private").sum()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
# Types des colonnes
df.dtypes

In [ ]:
# Valeurs nulles
df.isnull().sum()

## Étape 2 — Nettoyage

Quatre opérations :

1. extraire le ticker avec une regex (on ne garde que la 1re valeur quand plusieurs cotations sont listées) ;
2. supprimer les sociétés privées et les associations ;
3. fusionner les sociétés partageant un même ticker, en agrégeant leurs couches de supply chain ;
4. supprimer la colonne `company`, récupérable plus tard via yfinance.

**139 → 102 lignes.**

In [ ]:
# Load the CSV file in a DataFrame.
df = pd.read_csv(RAW_CSV)

In [ ]:
# Extract the ticker (only keep the first if multiple values are present, usually due to multiple cotations)
df['ticker'] = df['ticker'].str.extract(r'^([^(/]+)', expand=False).str.strip()
df.head(30)

Remove non publicly traded companies.

In [ ]:
remove_private = (df["ticker"].isin(["Private", "Non-Profit"]) == False)
df_private = df[remove_private].reset_index(drop=True)
df_private["ticker"] = df_private["ticker"].str.strip().str.upper()
df_private

Merge the companies sharing the same ticker

In [ ]:
df_private["layer_ids"] = df_private["Layer"].str.findall(r"(\d+)").apply(lambda ids: {int(i) for i in ids})
# Layer ids are merged and sorted numerically (10 before 3), not lexicographically.
df_merged = df_private.groupby("ticker", as_index=False).agg(
    layer_ids=("layer_ids", lambda ids: sorted({i for lst in ids for i in lst})),
    company=("company", lambda names: " / ".join(sorted(set(names)))),
)
df_merged["layer"] = df_merged["layer_ids"].apply(lambda ids: ", ".join(f"Layer {i}" for i in ids))
df_merged = df_merged.drop(columns="layer_ids").reset_index(drop=True)
df_merged

Removing company names as they can be found later with the yfinance library via the ticker.

In [ ]:
df_cleaned = df_merged.drop(["company"], axis=1)
df_cleaned

Create a simple dataset with only the tickers.

In [ ]:
df_tickers = df_cleaned.drop(["layer"], axis=1)
df_tickers

# Export

In [ ]:
df_cleaned.to_csv(OUT_DIR / 'stocks_cleaned.csv', index=False)
df_tickers.to_csv(OUT_DIR / 'tickers.csv', index=False)

## Étape 3 — Résolution des tickers sur yfinance

Certains tickers ne sont pas reconnus par yfinance (généralement les bourses non américaines) : il faut leur ajouter le suffixe de la place de marché. Plutôt que de lister les tickers un par un, on **déduit la place de cotation à partir de la longueur du ticker** et on teste les candidats par ordre de priorité.

**102 → 100 lignes.** ⏱️ Cette étape est la plus longue du notebook (une requête par candidat).

Some tickers are not recognized by yfinance (usually non-US stocks) and need to be modified to add the country suffix at the end of the ticker. In order to optimize the tests, we use patterns to try to deduce ASAP the quotation country.

In [ ]:
# 1. Read CSV while forcing the ticker column to stay as text (keeps leading zeros)
df = pd.read_csv(OUT_DIR / "stocks_cleaned.csv", dtype={"ticker": str})
tickers = df["ticker"].dropna().tolist()

In [ ]:
def find_valid_ticker(raw_ticker):
    
    # 2. Find the tickers working with yfinance library.
    ticker = str(raw_ticker).strip().upper()
    candidates = []

    # Check numeric-only tickers based on digit length
    if ticker.isdigit():
        digit_count = len(ticker)

        if digit_count == 4:
            # 4-digit tickers: Try 4-digit markets first (Japan, Taiwan, Hong Kong)
            ticker_4 = ticker.zfill(4)
            candidates.append(ticker_4 + ".T")   # Japan
            candidates.append(ticker_4 + ".TW")  # Taiwan TWSE
            candidates.append(ticker_4 + ".TWO") # Taiwan TPEx
            candidates.append(ticker_4 + ".HK")  # Hong Kong

            # Fallback: Zero-pad to 6 digits for Korea / China
            ticker_6 = ticker.zfill(6)
            candidates.append(ticker_6 + ".KS")  # Korea KOSPI
            candidates.append(ticker_6 + ".KQ")  # Korea KOSDAQ
            candidates.append(ticker_6 + ".SS")  # Shanghai
            candidates.append(ticker_6 + ".SZ")  # Shenzhen

        elif digit_count == 6:
            # 6-digit tickers: Try 6-digit markets FIRST (Korea, China)
            candidates.append(ticker + ".KS")  # Korea KOSPI
            candidates.append(ticker + ".KQ")  # Korea KOSDAQ
            candidates.append(ticker + ".SS")  # Shanghai
            candidates.append(ticker + ".SZ")  # Shenzhen

            # Fallback: Trim to 4 digits for Japan / Taiwan
            ticker_4 = ticker[-4:]
            candidates.append(ticker_4 + ".T")   # Japan
            candidates.append(ticker_4 + ".TW")  # Taiwan
            candidates.append(ticker_4 + ".HK")  # Hong Kong

        else:
            # 5-digit or other lengths (e.g. Hong Kong 1810 / 9880)
            ticker_4 = ticker.zfill(4)
            ticker_6 = ticker.zfill(6)
            candidates.append(ticker_4 + ".HK")
            candidates.append(ticker_6 + ".KS")
            candidates.append(ticker_6 + ".SS")
            candidates.append(ticker_6 + ".SZ")

    else:
        # Non-numeric tickers (US & popular European tickers)
        candidates.append(ticker)             # US
        candidates.append(ticker + ".DE")     # Germany
        candidates.append(ticker + ".F")      # Frankfurt
        candidates.append(ticker + ".L")      # London
        candidates.append(ticker + ".PA")     # Paris
        candidates.append(ticker + ".SW")     # Switzerland

    # Remove duplicates while keeping exact order of priority
    candidates = list(dict.fromkeys(candidates))

    # Test candidate tickers to see if a yfinance command works (valid ticker):
    for candidate in candidates:
        try:
            hist = yf.Ticker(candidate).history(period="1d")
            if not hist.empty:
                print(f"SUCCESS: {raw_ticker} -> {candidate}")
                return candidate
        except Exception:
            pass

    print(f"NOT FOUND: {raw_ticker}")


In [ ]:
# 2. run over all tickers and collect results
results = {}
not_found_tickers = []

for t in tickers:
    matched = find_valid_ticker(t)
    results[t] = matched
    
    # Collect missing tickers into a list
    if matched is None:
        not_found_tickers.append(t)

print("Finished processing all tickers.")

In [ ]:
# 3. Print the missing tickers summary at the end
print(f"SUMMARY: {len(not_found_tickers)} ticker(s) could not be resolved:")
if not_found_tickers:
    print(not_found_tickers)
else:
    print("All tickers were successfully matched!")

In [ ]:
# 4. Update ticker column with resolved tickers
df["ticker"] = df["ticker"].replace(results)

# Remove rows where no valid ticker was found
df = df.dropna(subset=["ticker"])
df

# Export

In [ ]:
# Export
df.to_csv(OUT_DIR / "stocks_transformed_verified.csv", index=False)

## Étape 4 — Construction des KPIs

Deux KPIs complémentaires pour chaque ticker :

- le **PEG Ratio (1Y)** : valorisation relative à la croissance attendue ;
- le **ROIC (%)** : efficacité du capital immobilisé, `NOPAT / Invested Capital` avec `NOPAT = EBIT × (1 - tax rate)` et `Invested Capital = Total Debt + Stockholders Equity - Cash`.

⏱️ Nouvelle série d'appels réseau : les tickers pour lesquels on n'obtient pas les deux valeurs sont écartés par le `try/except`. Le dataframe filtré (`df_ok`) exclut ensuite les PEG négatifs, les PEG > 10 et les lignes incomplètes.

**100 → 58 lignes.** La médiane du ROIC est calculée sur ce dataframe filtré, puis chaque ticker restant est classé dans l'un des quatre quadrants PEG × ROIC.

In [ ]:
# Create a list of tickers
df = pd.read_csv(OUT_DIR / "stocks_transformed_verified.csv")
tickers = df["ticker"].tolist()
tickers

In [ ]:
# I create a list to store the KPI data for each ticker. I will create a DataFrame from this list at the end of the loop.
kpi_data = []

In [ ]:
# Calculate KPIs for each stock.
for ticker in tickers:

    try:
        t = yf.Ticker(ticker)
        
        # KPI #1: PEG ratio.
        forward_EPS_growth = ((t.info['forwardEps'] - t.info['epsCurrentYear']) / t.info['epsCurrentYear']) * 100
        PEG_ratio = t.info['forwardPE'] / forward_EPS_growth

        # KPI #2: ROIC (ROIC = NOPAT / Invested Capital
        # 
        # --> NOPAT = EBIT * (1 - tax rate)
        # --> Invested Capital = Total Debt + Stockholders Equity - Cash and Cash Equivalents

        ebit = t.financials.loc['EBIT'].iloc[0]
        tax_Rate = t.financials.loc["Tax Rate For Calcs"].iloc[0]
        nopat = ebit * (1 - tax_Rate)

        tot_Debt = t.balance_sheet.loc["Total Debt"].iloc[0]
        tot_Equity = t.balance_sheet.loc["Stockholders Equity"].iloc[0]
        cash = t.balance_sheet.loc["Cash And Cash Equivalents"].iloc[0]
        invested_Capital = tot_Debt + tot_Equity - cash

        roic = (nopat / invested_Capital) * 100


        # I also get the company name
        name = t.info['shortName']


        # Add the KPIs to the list.
        kpi_data.append({
            "Ticker": ticker,
            "PEG Ratio (1Y)": round(PEG_ratio, 2),
            "ROIC (%)": round(roic, 2),
            "Name": name
        })

    except Exception as e:
        print(f"Could not get data for {ticker}: {e}")

In [ ]:
df_kpis = pd.DataFrame(kpi_data)
df_kpis

## Classification

Je classe chaque ticker dans une catégorie pour regrouper des tickers aux caractéristiques similaires.

- Efficace & Sous-évaluée (ES) : PEG ≤ 1 + ROIC ≥ médiane
- Efficace & Valuée (EV) : PEG > 1 + ROIC ≥ médiane
- Peu efficace & Sous-évaluée (PS) : PEG ≤ 1 + ROIC < médiane
- Peu efficace & Valuée (PV) : PEG > 1 + ROIC < médiane

Inclassables : tickers avec un PEG > 10 ou < 0.

In [ ]:
# Masks
peg_positif = df_kpis["PEG Ratio (1Y)"] > 0
peg_less_than_10 = df_kpis["PEG Ratio (1Y)"] <= 10
has_peg = df_kpis["PEG Ratio (1Y)"].notna()
has_roic = df_kpis["ROIC (%)"].notna()

# Filter the dataframe to remove outliers and tickers missing data.
df_ok = df_kpis[(peg_positif) & (peg_less_than_10) & (has_peg) & (has_roic)]

# Calculate the median ROIC.
roic_median = round(df_ok["ROIC (%)"].median(), 2)
roic_median

In [ ]:
# Assign a category to each ticker.
def get_category(row):
    peg = row["PEG Ratio (1Y)"]
    roic = row["ROIC (%)"]

    if peg <= 1 and roic >= roic_median:
        return "ES"
    elif peg > 1 and roic >= roic_median:
        return "EV"
    elif peg <= 1 and roic < roic_median:
        return "PS"
    else:
        return "PV"


df_ok["Category"] = df_ok.apply(get_category, axis=1)
df_ok

## Export

In [ ]:
# Export
df_ok.to_csv(OUT_DIR / "stocks_kpis.csv", index=False)

## Étape 5 — Visualisation

Aucune sélection supplémentaire ici : les 58 sociétés retenues à l'étape 4 sont toutes affichées, colorées par quadrant.

Le graphique est découpé par deux frontières : `PEG = 1` (valorisation) et la médiane du ROIC de l'échantillon (efficacité du capital).

In [ ]:
# Load KPI data
df_scatter = pd.read_csv(OUT_DIR / "stocks_kpis.csv")

Create the graph:

In [ ]:
# Create scatter plot
fig = px.scatter(
    df_scatter,
    x="PEG Ratio (1Y)",
    y="ROIC (%)",
    color="Category",
    text="Ticker",
    hover_data=["Name"],
    width=1200,
    height=675
)

# Separate the graph in sections.
roic_median = df_scatter["ROIC (%)"].median()
fig.add_vline(x=1, line_dash="solid", annotation_text="PEG = 1")
fig.add_hline(y=roic_median, line_dash="solid", annotation_text=f"ROIC médian = {roic_median:.1f}%")

fig.update_layout(legend_title_text="Category")

fig.show()

- **ES** (upper-left): High ROIC + Low PEG = efficient companies at relatively attractive valuations.
- **EV** (upper-right): High ROIC + High PEG = strong companies, but relatively expensive.
- **PS** (lower-left): Low ROIC + Low PEG = relatively cheap, but lower capital efficiency.
- **PV** (lower-right): Low ROIC + High PEG = lower efficiency and relatively expensive.

> Le nombre de sociétés perdues à chaque étape est détaillé dans `06_Recap.ipynb`.